In [12]:
import pandas as pd
from collections import defaultdict

def create_document_dataframe(doc_ids_path, lang_files):
    """
    Builds a DataFrame from document IDs and multilingual text files.

    Args:
        doc_ids_path (str): The file path for the TSV file containing document IDs.
        lang_files (dict): A dictionary where keys are language codes (e.g., 'text_fr')
                           and values are the corresponding text file paths.

    Returns:
        pandas.DataFrame: A DataFrame with columns "Document_id" and one column
                          for each language's concatenated text.
    """
    # --- 1. Read Document IDs ---

    doc_ids = pd.read_csv(doc_ids_path, sep='\t', header=None, names=['Document_id'])
    
    # --- 2. Process Language Files and Concatenate Sentences ---
    document_texts = defaultdict(lambda: {lang: [] for lang in lang_files.keys()})

    # Read each language file line by line
    for lang_code, file_path in lang_files.items():
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                sentences = [line.strip() for line in f.readlines()]
        except FileNotFoundError:
            print(f"Error: The file {file_path} was not found.")
    
        # Ensure the number of sentences matches the number of document IDs
        if len(sentences) != len(doc_ids):
            raise ValueError(
                f"The number of lines in {file_path} ({len(sentences)}) does not match "
                f"the number of document IDs ({len(doc_ids)})."
            )

        # Associate each sentence with its corresponding document ID
        for i, doc_id in enumerate(doc_ids['Document_id']):
            document_texts[doc_id][lang_code].append(sentences[i])

    # --- 3. Create the Final DataFrame ---
    processed_data = []
    for doc_id, lang_map in document_texts.items():
        row = {'Document_id': doc_id}
        for lang_code, sentences_list in lang_map.items():
            row[lang_code + "_list"] = sentences_list
            row[lang_code] = ' '.join(sentences_list)
        processed_data.append(row)
    print(processed_data[0])
    # Convert the list of dictionaries to a DataFrame
    final_df = pd.DataFrame(processed_data)

    # Reorder columns to have Document_id first
    # column_order = ['Document_id'] + sorted(lang_files.keys())
    # final_df = final_df[column_order]

    return final_df


In [29]:
path="/home/yathagata/NTREX/"


document_ids_file = path + "DOCUMENT_IDS.tsv"
language_files = {
    "text_fr": path + "NTREX-128/newstest2019-ref.fra.txt",
    "text_en": path + "NTREX-128/newstest2019-ref.eng-GB.txt",
    "text_de": path + "NTREX-128/newstest2019-ref.deu.txt",
    "text_es": path + "NTREX-128/newstest2019-ref.spa.txt",
    "text_ar": path + "NTREX-128/newstest2019-ref.arb.txt",
    "text_jp": path + "NTREX-128/newstest2019-ref.jpn.txt",
    # "text_tir": path + "NTREX-128/newstest2019-ref.tir.txt",
    # "text_eus": path + "NTREX-128/newstest2019-ref.eus.txt",
    # "text_dzo": path + "NTREX-128/newstest2019-ref.dzo.txt",
    # "text_mri": path + "NTREX-128/newstest2019-ref.mri.txt",
    # "text_khm": path + "NTREX-128/newstest2019-ref.khm.txt",
}

my_dataframe = create_document_dataframe(document_ids_file, language_files)

print("Successfully created the DataFrame:")
print(my_dataframe)

{'Document_id': 'bbc.381790', 'text_fr_list': ['Des membres de l’Assemblée du pays de Galles inquiets de «\xa0passer pour des marionnettes\xa0»', 'Des élus gallois sont consternés par une proposition visant à remplacer leur titre par le sigle MWP (Member of the Welsh Parliament, Membre du Parlement gallois).', 'Cette proposition découle du projet de modification du nom de l’assemblée, qui deviendrait le Parlement gallois.', 'Dans la sphère politique, des membres de l’Assemblée pensent que cela pourrait faire l’objet de railleries.', 'Un élu travailliste a indiqué que son groupe est préoccupé par le fait que le sigle «\xa0rime avec Twp et Pwp\xa0».', 'Pour les lecteurs non originaires du Pays de Galles\xa0: en gallois, twp signifie stupide, et pwp signifie crotte.', 'Selon l’un de ses membres, le Plaid Cymru (Parti du Pays de Galles) n’est pas «\xa0satisfait\xa0» et a proposé d’autres formules.', 'Le parti conservateur serait «\xa0ouvert\xa0» à un changement de nom, mais a fait remarque

In [30]:
from datasets import Dataset, DatasetDict

hf_dataset = Dataset.from_pandas(my_dataframe)

In [31]:
hf_dataset

Dataset({
    features: ['Document_id', 'text_fr_list', 'text_fr', 'text_en_list', 'text_en', 'text_de_list', 'text_de', 'text_es_list', 'text_es', 'text_ar_list', 'text_ar', 'text_jp_list', 'text_jp'],
    num_rows: 123
})

In [33]:
res = DatasetDict()
for language in ["fr", "de", "es", "ar", "jp"]:
    res[f"en-{language}"] = hf_dataset.map(
        lambda x: {
            "id": x["Document_id"],
            "src": x["text_en"],
            "trg": x[f"text_{language}"],
            "src_list": x["text_en_list"],
            "trg_list": x[f"text_{language}_list"]
        },
        remove_columns=[col for col in hf_dataset.column_names if col not in ["id", "src", "trg", "src_list", "trg_list"]]
    )



Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

In [36]:
res["en-es"][100]

{'id': 'abcnews.306774',
 'src': 'Federal prosecutors seeking rare death penalty for NYC terror attack suspect Federal prosecutors in New York are seeking the death penalty for Sayfullo Saipov, the suspect in the New York City terror attack that killed eight people -- a rare punishment that hasn\'t been carried out in the state for a federal crime since 1953. Saipov, 30, allegedly used a Home Depot rental truck to carry out an attack on a bike path along the West Side Highway in Lower Manhattan, mowing down pedestrians and cyclist in his path on Oct. In order to justify a death sentence, prosecutors will have to prove that Saipov "intentionally" killed the eight victims and "intentionally" inflicted serious bodily injury, according to the notice of intent to seek the death penalty, filed in the Southern District of New York. Both of those counts carry a possibly death sentence, according to the court document. Weeks after the attack, a federal grand jury slapped Saipov with a 22-count 

In [37]:
res.save_to_disk("/data/cef/ntrex/ntrex_original")

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

In [1]:
from datasets import load_from_disk

# Load the dataset from disk
dataset = load_from_disk("/data/cef/NTREX")

# Print the first few rows of the dataset
print(dataset[:5])

{'Document_id': ['bbc.381790', 'rt.com.91337', 'nytimes.184853', 'upi.176266', 'guardian.221754'], 'text_de': ['Mitglieder der Nationalversammlung für Wales befürchten Verwechslungen mit den Muppets Bei einigen Nationalversammlungsmitgliedern löst der Vorschlag, ihren Titel in „MWP“ (Member of the Welsh Parliament, Mitglied des walisischen Parlaments) zu ändern, Bestürzung aus. Grund dafür sind Pläne, den Namen der Nationalversammlung in „Welsh Parliament“ (Walisisches Parlament) zu ändern. Nationalversammlungsmitglieder aller politischen Richtungen befürchten, zur Zielscheibe des Spotts zu werden. Ein der Labour Party angehörendes Nationalversammlungsmitglied sagte, seine Gruppe sei besorgt, weil sich der Name „mit ‚Twp‘ und ‚Pwp‘ reimt“. Für nichtwalisische Leser: Im Walisischen steht „twp“ für „doof“ und „pwp“ für „Aa“. Ein Nationalversammlungsmitglied der Partei Plaid Cymru gab an, dass die gesamte Gruppe „nicht glücklich“ sei und Alternativen vorgeschlagen habe. Ein Mitglied der W

In [2]:
print(dataset[0])

{'Document_id': 'bbc.381790', 'text_de': 'Mitglieder der Nationalversammlung für Wales befürchten Verwechslungen mit den Muppets Bei einigen Nationalversammlungsmitgliedern löst der Vorschlag, ihren Titel in „MWP“ (Member of the Welsh Parliament, Mitglied des walisischen Parlaments) zu ändern, Bestürzung aus. Grund dafür sind Pläne, den Namen der Nationalversammlung in „Welsh Parliament“ (Walisisches Parlament) zu ändern. Nationalversammlungsmitglieder aller politischen Richtungen befürchten, zur Zielscheibe des Spotts zu werden. Ein der Labour Party angehörendes Nationalversammlungsmitglied sagte, seine Gruppe sei besorgt, weil sich der Name „mit ‚Twp‘ und ‚Pwp‘ reimt“. Für nichtwalisische Leser: Im Walisischen steht „twp“ für „doof“ und „pwp“ für „Aa“. Ein Nationalversammlungsmitglied der Partei Plaid Cymru gab an, dass die gesamte Gruppe „nicht glücklich“ sei und Alternativen vorgeschlagen habe. Ein Mitglied der Welsh Conservative Party sagte, seine Gruppe sei gegenüber der Namensän

In [3]:
print(dataset)

Dataset({
    features: ['Document_id', 'text_de', 'text_en', 'text_fr', 'text_sp'],
    num_rows: 123
})


In [4]:
europarl_example = load_from_disk("/data/cef/europarl/filtered_1381/en-de")
print(europarl_example)

Dataset({
    features: ['id', 'src', 'trg', 'src_tokens', 'trg_tokens', 'total_tokens'],
    num_rows: 1381
})


In [5]:
from datasets import load_from_disk, Dataset
import os

# Load the original dataset
dataset = load_from_disk("/data/cef/NTREX")

# Define the translation pairs
translation_pairs = [
    ("en-de", "text_en", "text_de"),
    ("en-fr", "text_en", "text_fr"), 
    ("en-es", "text_en", "text_sp")
]

# Create and save each translation dataset
for pair_name, src_lang, trg_lang in translation_pairs:
    print(f"Creating {pair_name} dataset...")
    
    # Create the translation dataset
    translation_data = []
    for i in range(len(dataset)):
        translation_data.append({
            'id': dataset[i]['Document_id'],
            'src': dataset[i][src_lang],
            'trg': dataset[i][trg_lang]
        })
    
    # Create the dataset
    translation_dataset = Dataset.from_list(translation_data)
    
    # Create directory if it doesn't exist
    save_dir = f"/data/cef/{pair_name}"
    os.makedirs(save_dir, exist_ok=True)
    
    # Save the dataset
    translation_dataset.save_to_disk(save_dir)
    
    print(f"✅ {pair_name} dataset saved to {save_dir}")
    print(f"   Features: {translation_dataset.features}")
    print(f"   Number of rows: {len(translation_dataset)}")
    print(f"   Sample: {translation_dataset[0]}")
    print()

print("🎉 All translation datasets created successfully!")

Creating en-de dataset...


Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

✅ en-de dataset saved to /data/cef/en-de
   Features: {'id': Value(dtype='string', id=None), 'src': Value(dtype='string', id=None), 'trg': Value(dtype='string', id=None)}
   Number of rows: 123
   Sample: {'id': 'bbc.381790', 'src': 'Welsh AMs worried about \'looking like muppets\' There is consternation among some AMs at a suggestion their title should change to MWPs (Member of the Welsh Parliament). It has arisen because of plans to change the name of the assembly to the Welsh Parliament. AMs across the political spectrum are worried it could invite ridicule. One Labour AM said his group was concerned "it rhymes with Twp and Pwp." For readers outside of Wales: In Welsh twp means daft and pwp means poo. A Plaid AM said the group as a whole was "not happy" and has suggested alternatives. A Welsh Conservative said his group was "open minded" about the name change, but noted it was a short verbal hop from MWP to Muppet. In this context The Welsh letter w is pronounced similarly to the Yo

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

✅ en-fr dataset saved to /data/cef/en-fr
   Features: {'id': Value(dtype='string', id=None), 'src': Value(dtype='string', id=None), 'trg': Value(dtype='string', id=None)}
   Number of rows: 123
   Sample: {'id': 'bbc.381790', 'src': 'Welsh AMs worried about \'looking like muppets\' There is consternation among some AMs at a suggestion their title should change to MWPs (Member of the Welsh Parliament). It has arisen because of plans to change the name of the assembly to the Welsh Parliament. AMs across the political spectrum are worried it could invite ridicule. One Labour AM said his group was concerned "it rhymes with Twp and Pwp." For readers outside of Wales: In Welsh twp means daft and pwp means poo. A Plaid AM said the group as a whole was "not happy" and has suggested alternatives. A Welsh Conservative said his group was "open minded" about the name change, but noted it was a short verbal hop from MWP to Muppet. In this context The Welsh letter w is pronounced similarly to the Yo

Saving the dataset (0/1 shards):   0%|          | 0/123 [00:00<?, ? examples/s]

✅ en-es dataset saved to /data/cef/en-es
   Features: {'id': Value(dtype='string', id=None), 'src': Value(dtype='string', id=None), 'trg': Value(dtype='string', id=None)}
   Number of rows: 123
   Sample: {'id': 'bbc.381790', 'src': 'Welsh AMs worried about \'looking like muppets\' There is consternation among some AMs at a suggestion their title should change to MWPs (Member of the Welsh Parliament). It has arisen because of plans to change the name of the assembly to the Welsh Parliament. AMs across the political spectrum are worried it could invite ridicule. One Labour AM said his group was concerned "it rhymes with Twp and Pwp." For readers outside of Wales: In Welsh twp means daft and pwp means poo. A Plaid AM said the group as a whole was "not happy" and has suggested alternatives. A Welsh Conservative said his group was "open minded" about the name change, but noted it was a short verbal hop from MWP to Muppet. In this context The Welsh letter w is pronounced similarly to the Yo

In [6]:
enes = load_from_disk("/data/cef/en-es")
print(enes)

Dataset({
    features: ['id', 'src', 'trg'],
    num_rows: 123
})
